# 02g · CNN benchmark: every model, same images, same metrics

Reads the saved predictions of 02a-02f (no training, ~2 min). For each model and fold, a temperature is fitted on
that fold's **validation** logits (temperature scaling never changes the predicted class). Then out-of-fold (OOF)
test predictions are pooled: every byte-unique image is scored exactly once.

Outputs: `results/cnn/benchmark.json`, `results/cnn/decision.json`, `results/cnn/paper_cnn.json`,
figures in `results/figures/`, and the CNN part of `results/RESULTS.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import config, env
from common import metrics as M
from common.cv import folds_to_run, preds_path

config.ensure_dirs()
MODELS = {"R0": "ResNet-50, current recipe", "V2": "ResNet-50 V2 (ours)",
          "G1": "GoogLeNet TL (Anudeep K.S. et al.)", "Y1": "YOLOv8-cls (Selva Kumar et al.)",
          "D0": "Deployed weights (reference only)"}
FOLDS = folds_to_run()
LG = [f"logit_{c}" for c in range(len(config.CLASSES))]

## Load predictions, fit temperatures, pool out-of-fold

In [ ]:
oof, temps, fold_scores = {}, {}, {}
for m in MODELS:
    missing = [k for k in FOLDS for s in ("val", "test") if not preds_path(m, k, s).exists()]
    if missing:
        print(f"{m}: predictions missing for folds {sorted(set(missing))} -> skipped (run its notebook)")
        continue
    parts, temps[m], fold_scores[m] = [], [], []
    for k in FOLDS:
        val = pd.read_csv(preds_path(m, k, "val"))
        test = pd.read_csv(preds_path(m, k, "test"))
        T = M.fit_temperature(val[LG].to_numpy(), val["y_true"].to_numpy())
        temps[m].append(T)
        test["fold"], test["T"] = k, T
        parts.append(test)
        fold_scores[m].append(M.classification_metrics(test["y_true"].to_numpy(), test[LG].to_numpy().argmax(1))["macro_f1"])
    oof[m] = pd.concat(parts).sort_values("path").reset_index(drop=True)
present = list(oof)
assert present, "No model predictions found - run the 02a-02f notebooks first."
ref_paths = oof[present[0]]["path"]
for m in present:
    assert oof[m]["path"].equals(ref_paths), f"{m} was scored on different images"
print("models:", present, "| OOF images:", len(ref_paths), "| source groups:", oof[present[0]]["group_id"].nunique())

## Metrics

In [ ]:
results = {}
for m in present:
    d = oof[m]
    y, g = d["y_true"].to_numpy(), d["group_id"].to_numpy()
    logits = d[LG].to_numpy()
    raw = M.softmax(logits)
    cal = np.concatenate([M.softmax(d.loc[d.fold == k, LG].to_numpy(), d.loc[d.fold == k, "T"].iloc[0])
                          for k in FOLDS])
    ycal = np.concatenate([d.loc[d.fold == k, "y_true"].to_numpy() for k in FOLDS])
    pred = logits.argmax(1)
    cost_file = config.CNN_RESULTS / f"{m}_cost.json"
    summ_file = config.CNN_RESULTS / f"{m}_summary.json"
    cost = json.loads(cost_file.read_text()) if cost_file.exists() else {}
    summ = json.loads(summ_file.read_text()) if summ_file.exists() else {}
    minutes = [f.get("train_minutes") for f in summ.get("folds", {}).values() if f.get("train_minutes") is not None]
    results[m] = {
        "name": MODELS[m],
        **M.classification_metrics(y, pred),
        "fold_macro_f1_mean": float(np.mean(fold_scores[m])), "fold_macro_f1_std": float(np.std(fold_scores[m])),
        "ci_macro_f1": M.group_bootstrap_ci(y, pred, g, "macro_f1"),
        "ci_accuracy": M.group_bootstrap_ci(y, pred, g, "accuracy"),
        "ece_raw": M.ece(raw, y), "ece_calibrated": M.ece(cal, ycal), "temperatures": temps[m],
        "gate_raw": M.gate_stats(raw, y), "gate_calibrated": M.gate_stats(cal, ycal),
        "confusion": M.confusion(y, pred).tolist(),
        "cost": cost, "train_minutes_per_fold": float(np.mean(minutes)) if minutes else None,
    }

table = pd.DataFrame({m: {
    "Accuracy": r["accuracy"], "Macro-F1": r["macro_f1"],
    "Macro-F1 95% CI": f"{r['ci_macro_f1']['lo']:.3f}-{r['ci_macro_f1']['hi']:.3f}",
    "Fold mean±sd": f"{r['fold_macro_f1_mean']:.3f}±{r['fold_macro_f1_std']:.3f}",
    "Macro-P": r["macro_precision"], "Macro-R": r["macro_recall"], "Bal. acc": r["balanced_accuracy"],
    **{f"F1 {config.PRETTY[c]}": r["per_class_f1"][c] for c in config.CLASSES},
    "ECE raw": r["ece_raw"], "ECE cal.": r["ece_calibrated"],
    "Gate coverage": r["gate_calibrated"]["coverage"], "Acc. answered": r["gate_calibrated"]["accuracy_answered"],
    "Errors caught": r["gate_calibrated"]["share_errors_caught"],
    "Params (M)": r["cost"].get("params", np.nan) / 1e6, "GFLOPs": r["cost"].get("gflops", np.nan),
    "GPU ms": r["cost"].get("gpu_ms_median", np.nan), "CPU ms": r["cost"].get("cpu_ms_median", np.nan),
    "Train min/fold": r["train_minutes_per_fold"],
} for m, r in results.items()})
pd.set_option("display.float_format", "{:.4f}".format)
table

## Is V2 really better? (paired tests on the same images)
Paired group bootstrap of the macro-F1 difference (resampling source patches) and exact McNemar test.
McNemar treats images as independent, which near-duplicates violate, so the bootstrap is the primary test.

In [ ]:
paired = {}
if "V2" in oof:
    yv = oof["V2"]["y_true"].to_numpy()
    gv = oof["V2"]["group_id"].to_numpy()
    pv = oof["V2"][LG].to_numpy().argmax(1)
    for m in present:
        if m == "V2":
            continue
        pm = oof[m][LG].to_numpy().argmax(1)
        paired[m] = {"bootstrap_macro_f1": M.paired_bootstrap_diff(yv, pv, pm, gv),
                     "mcnemar": M.mcnemar(yv, pv, pm)}
pd.DataFrame({m: {"ΔmacroF1 (V2-m)": p["bootstrap_macro_f1"]["diff"],
                  "95% CI": f"{p['bootstrap_macro_f1']['lo']:.3f} to {p['bootstrap_macro_f1']['hi']:.3f}",
                  "significant": p["bootstrap_macro_f1"]["significant"],
                  "McNemar p": p["mcnemar"]["p_value"]} for m, p in paired.items()}).T

## Replacement decision (rule fixed in PLAN.md §2.5 before any results)

In [ ]:
decision = {"available": False}
if {"V2", "R0", "D0"} <= set(results):
    v, r, d = results["V2"], results["R0"], results["D0"]
    b = paired["R0"]
    drops = {c: r["per_class_f1"][c] - v["per_class_f1"][c] for c in config.CLASSES}
    crit = {
        "1_better_macro_f1_significant": bool(v["macro_f1"] > r["macro_f1"] and
                                              (b["bootstrap_macro_f1"]["lo"] > 0 or b["mcnemar"]["p_value"] < 0.05)),
        "2_no_class_f1_drop_over_0.02": bool(max(drops.values()) <= 0.02),
        "3_calibrated_ece_not_worse": bool(v["ece_calibrated"] <= r["ece_calibrated"] + 1e-9),
        "4_cpu_latency_within_1.25x_deployed": bool(v["cost"].get("cpu_ms_median", np.inf)
                                                   <= 1.25 * d["cost"].get("cpu_ms_median", 0)),
    }
    decision = {"available": True, "criteria": crit, "replace": all(crit.values()), "class_f1_drop_vs_R0": drops,
                "v2_cpu_ms": v["cost"].get("cpu_ms_median"), "deployed_cpu_ms": d["cost"].get("cpu_ms_median")}
    for k_, ok in crit.items():
        print(("PASS " if ok else "FAIL ") + k_)
    print("\nDECISION:", "REPLACE the deployed model with V2 (run 02h)" if decision["replace"] else "KEEP the current model")
else:
    print("Need V2, R0 and D0 results for the decision.")

## Figures for the paper

In [ ]:
fair = [m for m in ("G1", "Y1", "R0", "V2") if m in results]
metric_names = [("accuracy", "Accuracy"), ("macro_f1", "Macro-F1"), ("macro_precision", "Macro-P"),
                ("macro_recall", "Macro-R")]
fig, ax = plt.subplots(figsize=(7.2, 4.0))
w = 0.8 / max(1, len(fair))
colors = {"G1": "#9ecae1", "Y1": "#6baed6", "R0": "#fdae6b", "V2": "#e6550d"}
for i, m in enumerate(fair):
    vals = [results[m][k] for k, _ in metric_names]
    xs = np.arange(len(metric_names)) + (i - (len(fair) - 1) / 2) * w
    ax.bar(xs, vals, w, label=MODELS[m], color=colors[m])
    ci = results[m]["ci_macro_f1"]
    ax.errorbar(xs[1], results[m]["macro_f1"], yerr=[[results[m]["macro_f1"] - ci["lo"]], [ci["hi"] - results[m]["macro_f1"]]],
                fmt="none", ecolor="black", capsize=3, lw=1)
ax.set_xticks(range(len(metric_names)), [n for _, n in metric_names])
lo = min(results[m][k] for m in fair for k, _ in metric_names)
ax.set_ylim(max(0, lo - 0.05), 1.0)
ax.set_ylabel("score (out-of-fold)")
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
ax.set_title(f"Module 1 on leak-free {len(FOLDS)}-fold CV (error bar: 95% group-bootstrap CI)", fontsize=9)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(config.FIG_DIR / f"fig_cnn_comparison.{ext}", dpi=200)
plt.show()

best = "V2" if "V2" in results else fair[-1]
cm = np.array(results[best]["confusion"], dtype=float)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
labels = [config.PRETTY[c] for c in config.CLASSES]
ax.set_xticks(range(5), labels, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(5), labels, fontsize=8)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cmn[i, j] > 0.6 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"{MODELS[best]}: row-normalised, out-of-fold", fontsize=8)
plt.tight_layout()
for ext in ("png", "pdf"):
    plt.savefig(config.FIG_DIR / f"fig_confusion.{ext}", dpi=200)
plt.show()

## Save

In [ ]:
abl_file = config.CNN_RESULTS / "v2_ablation.json"
ablation = json.loads(abl_file.read_text()) if abl_file.exists() else None
env.write_json(config.CNN_RESULTS / "benchmark.json",
               {"results": results, "paired_vs_V2": paired, "folds": FOLDS, "meta": env.run_meta()})
env.write_json(config.CNN_RESULTS / "decision.json", decision)

# same schema as IEEE-Paper/scripts/mock_data.json (paper order of confusion classes)
paper_order = ["Rust", "Cerscospora", "Phoma", "Miner", "Healthy"]
perm = [config.CLASSES.index(c) for c in paper_order]
paper = {
    "cnn_comparison": {"is_mock": False, "metrics": ["Accuracy", "Macro-F1", "Macro-Precision", "Macro-Recall"],
                       "systems": {MODELS[m]: [round(results[m][k], 4) for k, _ in metric_names] for m in fair}},
    "cnn_confusion": {"is_mock": False, "model": MODELS[best],
                      "classes": ["Rust", "Cercospora", "Phoma", "Miner", "Healthy"],
                      "matrix": np.array(results[best]["confusion"])[np.ix_(perm, perm)].tolist()},
    "table_vii": [{"method": MODELS[m], "acc": round(results[m]["accuracy"], 3),
                   "macro_f1": round(results[m]["macro_f1"], 3),
                   "phoma_f1": round(results[m]["per_class_f1"]["Phoma"], 3),
                   "params_m": round(results[m]["cost"].get("params", 0) / 1e6, 1),
                   "cpu_ms": round(results[m]["cost"].get("cpu_ms_median", float("nan")), 1)} for m in present],
    "v2_ablation": ablation["rows"] if ablation else None,
}
env.write_json(config.CNN_RESULTS / "paper_cnn.json", paper)

md = ["## Module 1 (CNN) - leak-free 5-fold cross-validation", "",
      f"Out-of-fold over {len(ref_paths):,} byte-unique images / {oof[present[0]]['group_id'].nunique()} source "
      f"patches. Sources: `results/cnn/benchmark.json`, predictions in `results/cnn/preds/`.", "",
      table.T.to_markdown(floatfmt=".4f"), ""]
if decision.get("available"):
    md += ["**Replacement decision:** " + ("replace with V2" if decision["replace"] else "keep current model"), ""]
    md += [f"- {k}: {'PASS' if v else 'FAIL'}" for k, v in decision["criteria"].items()] + [""]
if ablation:
    md += ["### V2 ablation (fold 0; decisions on validation only)", "",
           pd.DataFrame(ablation["rows"]).to_markdown(index=False, floatfmt=".4f"), ""]
(config.RESULTS_DIR / "RESULTS_cnn.md").write_text("\n".join(md), encoding="utf-8")
print("\n".join(md))